# Overview

This notebook is designed for the CatBoost model training.
We will use gradient boosting with extracted images embeddings in order to predict the litotypes on the source images.

Unfortunately, CatBoost doesn't natively support MPS, so CPU calculations will be used instead. 

## 1. Imports and Settings

In [1]:
import ast
import datetime
import os
import warnings

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [2]:
BASE_PATH = "/Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/gb/gb-training_lab/"

DATASET_PATH = "data/interim/"
DATASET_TARGET_FILE = "metadata_dinov3_embeddings.parquet"

FEATURE_COLS = [
    "interval_start", "interval_end"
]
TARGET_COLS = [
    'sandstone_sludge', 'siltstone_sludge', 'argillite_sludge'
]

FINAL_MODEL_IGNORED_WELL_ID = 1

OUTPUT_TYPE = "catboost"
OUTPUT_MODELS_PATH = "output/models/"
OUTPUT_METRICS_PATH = "output/metrics/"
MODEL_NAME = "catboost_model-{0}.cbm"
METRICS_NAME = "catboost-model-{0}-{1}.csv"

In [3]:
warnings.filterwarnings("ignore")

np.random.seed(42)

## 2. Dataset Loading

In [4]:
df = pd.read_parquet(os.path.join(BASE_PATH, DATASET_PATH, DATASET_TARGET_FILE))


print("Dataset loaded. Sample: ")
print(df.head())

Dataset loaded. Sample: 
   well_id  device_no  interval_start  interval_end  sandstone_sludge  \
0        4        1.0            2675          2680                 5   
1        4        2.0            2680          2685                 5   
2        4        3.0            2685          2690                 5   
3        4        4.0            2690          2695                 5   
4        4        5.0            2695          2700                 5   

   siltstone_sludge  argillite_sludge  radiolarite_sludge  coal_sludge  \
0                25                70                   0            0   
1                25                70                   0            0   
2                30                65                   0            0   
3                30                65                   0            0   
4                25                70                   0            0   

   limestone_sludge  ...  oil_saturation  calcite_carbonatometry  \
0                 0  ..

## 3. Embeddings Processing

In [5]:
def str_to_array(x):
    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x))
        except:
            return np.array(x)
    return x

df['lba_dinov3_emb'] = df['lba_dinov3_emb'].apply(str_to_array)
df['sludge_dinov3_emb'] = df['sludge_dinov3_emb'].apply(str_to_array)

print("Embeddings formatted as arrays.")
print(df['sludge_dinov3_emb'].values[0].shape)

Embeddings formatted as arrays.
(1024,)


## 4. Features and Targets Preparation

In [6]:
# We reduce 1024-dimensional embeddings to 32 principal components
N_PCA_COMPONENTS = 32

pca_lba = PCA(n_components = N_PCA_COMPONENTS, random_state = 42)
lba_emb_pca = pca_lba.fit_transform(df['lba_dinov3_emb'].tolist())

pca_sludge = PCA(n_components = N_PCA_COMPONENTS, random_state = 42)
sludge_emb_pca = pca_sludge.fit_transform(df['sludge_dinov3_emb'].tolist())

lba_emb_df = pd.DataFrame(
    lba_emb_pca,
    columns = [f"lba_emb_pca_{i}" for i in range(N_PCA_COMPONENTS)]
)
sludge_emb_df = pd.DataFrame(
    sludge_emb_pca,
    columns = [f"sludge_emb_pca_{i}" for i in range(N_PCA_COMPONENTS)]
)

input_df = pd.concat([df[FEATURE_COLS].reset_index(drop = True), sludge_emb_df, lba_emb_df], axis = 1)
print("Input features are prepared (with PCA).")
print(input_df.head())

output_df = df[TARGET_COLS].reset_index(drop = True)
print("Target variables are prepared.")
print(output_df.head())


Input features are prepared (with PCA).
   interval_start  interval_end  sludge_emb_pca_0  sludge_emb_pca_1  \
0            2675          2680          6.760004          4.925340   
1            2680          2685          4.876463          4.124293   
2            2685          2690          5.113024          4.624120   
3            2690          2695          4.909690          1.396425   
4            2695          2700          4.562015          2.188327   

   sludge_emb_pca_2  sludge_emb_pca_3  sludge_emb_pca_4  sludge_emb_pca_5  \
0         -2.098502         -1.877884         -1.794193          0.165442   
1         -1.333347         -4.041059         -1.776195          0.559525   
2         -1.221538         -3.517736         -1.912365          0.700864   
3         -0.482193         -4.905857         -1.418478          1.135350   
4         -0.648930         -5.419340         -0.273700          2.110142   

   sludge_emb_pca_6  sludge_emb_pca_7  ...  lba_emb_pca_22  lba_emb_pc

## 5. Training

### 5.1. Hyper-Params

In [ ]:
catboost_params = {
    'iterations': 5000,
    'learning_rate': 0.015,
    'depth': 5,
    'l2_leaf_reg': 10,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'rsm': 0.7,
    'loss_function': 'MultiRMSE',
    'eval_metric': 'MultiRMSE',
    'random_seed': 42,
    'early_stopping_rounds': 200,
    'verbose': 100,
    'task_type': 'CPU',
    'devices': '0'
}

### 5.2. Helpers Declaration

In [8]:
class TabularDataset:
    def __init__(self, X, y, groups = None):
        self.X = X
        self.y = y
        self.groups = groups

    def get_fold(self, train_idx, val_idx):
        X_train = self.X.iloc[train_idx]
        X_val = self.X.iloc[val_idx]

        y_train = self.y.iloc[train_idx]
        y_val = self.y.iloc[val_idx]

        return X_train, X_val, y_train, y_val

    def extract_folds(self, n_splits = 4):
        gkf = GroupKFold(n_splits = n_splits)
        folds = []
        for train_idx, val_idx in gkf.split(self.X, self.y, self.groups):
            X_train = self.X.iloc[train_idx]
            X_val = self.X.iloc[val_idx]
            y_train = self.y.iloc[train_idx]
            y_val = self.y.iloc[val_idx]
            folds.append((X_train, X_val, y_train, y_val))
        return folds


In [9]:
def normalize_predictions(predictions):
    clipped = np.clip(predictions, 0, None)
    predictions_sum = clipped.sum(axis = 1, keepdims = True)
    predictions_sum = np.where(predictions_sum == 0, 1.0, predictions_sum)
    return clipped / predictions_sum * 100

def train_model(X_train, y_train, X_val, y_val, params):
    model = CatBoostRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set = (X_val, y_val),
        use_best_model = True,
        verbose = True
    )
    return model

def train_model_no_eval(X_train, y_train, params):
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, verbose = False)
    return model

def save_model_to_disk(model, filepath):
    model.save_model(filepath)

def save_predictions_to_csv(predictions, y_true, filepath):
    pred_renamed = predictions.rename(columns = lambda x: f"{x}_pred")
    y_true_renamed = y_true.rename(columns = lambda x: f"{x}_true")
    combined_df = pd.concat([pred_renamed, y_true_renamed], axis = 1)
    combined_df.to_csv(filepath)

def get_predictions_by_model(model, X_val, y_val):
    pred = model.predict(X_val)
    pred = normalize_predictions(pred)
    pred_df = pd.DataFrame(
        pred,
        columns = y_val.columns,
        index = y_val.index
    )
    return pred_df

def calculate_metrics(
    y_true: pd.DataFrame,
    y_predicted: pd.DataFrame
) -> dict:
    metrics = {}
    for target in y_true.columns:
        metrics[target] = {
            "mae": mean_absolute_error(
                y_true[target],
                y_predicted[target]
            ),
            "rmse": root_mean_squared_error(
                y_true[target],
                y_predicted[target]
            ),
            "r2": r2_score(
                y_true[target],
                y_predicted[target]
            )
        }
    return metrics

def print_metrics(metrics):
    print("!== Cross-Validation Metrics ==!")
    for target in metrics:
        print(target)
        # MAE:
        print(
            f"MAE: {np.mean(metrics[target]['mae']):.4f} "
            f"+/- {np.std(metrics[target]['mae']):.4f}"
        )
        # RMSE:
        print(
            f"RMSE: {np.mean(metrics[target]['rmse']):.4f} "
            f"+/- {np.std(metrics[target]['rmse']):.4f}"
        )
        # R2:
        print(
            f"R2: {np.mean(metrics[target]['r2']):.4f} "
            f"+/- {np.std(metrics[target]['r2']):.4f}"
        )
        print("===")


### 5.3. Orchestration Functions

In [10]:
def run_cv(
    input_df,
    output_df,
    groups,
    params,
    n_splits = 4
):
    dataset = TabularDataset(input_df, output_df, groups)
    folds = dataset.extract_folds(n_splits = n_splits)
    
    aggregated_metrics = {
        target: {
            "mae": [],
            "rmse": [],
            "r2": []
        }
        for target in output_df.columns
    }
    
    # Ensure directory for metrics exists
    os.makedirs(os.path.join(BASE_PATH, OUTPUT_METRICS_PATH, OUTPUT_TYPE), exist_ok = True)
    
    for fold_idx, (X_train, X_val, y_train, y_val) in enumerate(folds):
        model = train_model(X_train, y_train, X_val, y_val, params)
        predictions = get_predictions_by_model(model, X_val, y_val)
        
        # Save predictions to csv with original values side-by-side
        current_datetime = datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
        filename = METRICS_NAME.format(fold_idx + 1, current_datetime)
        filepath = os.path.join(BASE_PATH, OUTPUT_METRICS_PATH, OUTPUT_TYPE, filename)
        save_predictions_to_csv(predictions, y_val, filepath)
        
        fold_metrics = calculate_metrics(y_val, predictions)
        
        for target in fold_metrics:
            for metric_name in fold_metrics[target]:
                aggregated_metrics[target][metric_name].append(
                    fold_metrics[target][metric_name]
                )
                
        print(f"Fold {fold_idx + 1} completed.")
        
    return aggregated_metrics

def run_final_training(
    input_df,
    output_df,
    groups,
    params
):
    train_mask = groups != FINAL_MODEL_IGNORED_WELL_ID
    test_mask = groups == FINAL_MODEL_IGNORED_WELL_ID

    X_train = input_df[train_mask]
    y_train = output_df[train_mask]
    X_test = input_df[test_mask]
    y_test = output_df[test_mask]

    print(f"Training final model on wells: {groups[train_mask].unique()}.")
    print(f"Testing final model on holdout well: {FINAL_MODEL_IGNORED_WELL_ID} ({len(X_test)} samples).")

    # 1. Train final model
    model = train_model_no_eval(X_train, y_train, params)

    # 2. Save final model
    current_datetime = datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
    model_name = MODEL_NAME.format(current_datetime)
    model_output_path = os.path.join(BASE_PATH, OUTPUT_MODELS_PATH, OUTPUT_TYPE, model_name)
    save_model_to_disk(model, model_output_path)
    print(f"Holdout model saved to: {model_output_path}.")

    # 3. Get predictions and normalize
    test_predictions = get_predictions_by_model(model, X_test, y_test)

    # 4. Save predictions side-by-side with true values to CSV
    test_filename = METRICS_NAME.format(current_datetime, "Final")
    test_filepath = os.path.join(BASE_PATH, OUTPUT_METRICS_PATH, OUTPUT_TYPE, test_filename)
    save_predictions_to_csv(test_predictions, y_test, test_filepath)
    print(f"Holdout predictions saved to: {test_filepath}.")

    # 5. Calculate metrics on the holdout well
    test_metrics = calculate_metrics(y_test, test_predictions)
    print_metrics(test_metrics)

    return model

### 5.4. Training / Saving

In [ ]:
groups = df["well_id"]
metrics = run_cv(
    input_df,
    output_df,
    groups,
    catboost_params,
    n_splits = 4
)
print_metrics(metrics)

0:	learn: 40.0576703	test: 77.4855489	best: 77.4855489 (0)	total: 58.4ms	remaining: 24m 20s
1:	learn: 40.0173965	test: 77.5083090	best: 77.4855489 (0)	total: 61.3ms	remaining: 12m 45s
2:	learn: 39.9791915	test: 77.5197196	best: 77.4855489 (0)	total: 64ms	remaining: 8m 53s
3:	learn: 39.9352085	test: 77.5295122	best: 77.4855489 (0)	total: 66ms	remaining: 6m 52s
4:	learn: 39.8944705	test: 77.5322243	best: 77.4855489 (0)	total: 68.1ms	remaining: 5m 40s
5:	learn: 39.8541175	test: 77.5449599	best: 77.4855489 (0)	total: 70ms	remaining: 4m 51s
6:	learn: 39.8150911	test: 77.5436888	best: 77.4855489 (0)	total: 72ms	remaining: 4m 16s
7:	learn: 39.7742271	test: 77.5530277	best: 77.4855489 (0)	total: 74.2ms	remaining: 3m 51s
8:	learn: 39.7370805	test: 77.5612592	best: 77.4855489 (0)	total: 76.2ms	remaining: 3m 31s
9:	learn: 39.7007900	test: 77.5737643	best: 77.4855489 (0)	total: 78.1ms	remaining: 3m 15s
10:	learn: 39.6622379	test: 77.5862193	best: 77.4855489 (0)	total: 80ms	remaining: 3m 1s
11:	lea

In [ ]:
groups = df["well_id"]
holdout_model = run_final_training(
    input_df,
    output_df,
    groups,
    catboost_params
)

Training final model on wells: [4 2 3]
Testing final model on holdout well: 1 (390 samples)
Holdout model saved to: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/gb/gb-training_lab/output/models/catboost/catboost_model_holdout_well_1.cbm
Holdout predictions saved to: /Users/locked15/Repos.localized/Python/Study/UUST/Course 3/Semester 2/Practice/sludge-utilities/apps/gb/gb-training_lab/output/metrics/catboost/catboost-model-holdout-well-1-2026-06-15_14:13:02.csv

!== Holdout Test Well Metrics ==!
sandstone_sludge:
  MAE : 11.2513
  RMSE: 13.7546
  R2  : 0.8495
siltstone_sludge:
  MAE : 4.0908
  RMSE: 5.2547
  R2  : 0.8418
argillite_sludge:
  MAE : 10.2926
  RMSE: 12.4456
  R2  : 0.7272
